# 🛡️ Notebook 03 — Data Quality Framework

**Goal:** Build a reusable data quality validation framework. Quarantine bad records to a dead-letter table. Never let dirty data reach Gold.

> **Run time:** ~5 min

## Architecture
```
Incoming Data
    │
    ▼
┌─────────────────────┐
│  Quality Rules       │  ← define rules as code
│  ─ NOT NULL checks  │
│  ─ Range validation │
│  ─ Enum validation  │
│  ─ FK integrity     │
│  ─ Duplicate check  │
└──────┬──────────────┘
       │
  ┌────┴─────┐
  ▼           ▼
PASS        FAIL
  │           │
Silver      dead_letter_transactions
(clean)     (quarantine + reason)
```

In [ ]:
# Import PySpark functions and schema types used to evaluate data quality rules
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Read the intentionally dirty transaction file that contains records to validate
df_dirty = spark.read.option('header','true').option('inferSchema','true')     .csv('Files/dirty_transactions.csv')

# Preview the incoming dataset before applying any quality checks
print(f'Incoming records: {df_dirty.count()}')
df_dirty.show(truncate=False)

## Step 1 — Define Quality Rules

In [ ]:
# Define the allowed business values and threshold limits for the quality checks
valid_txn_types = ['Deposit','Withdrawal','Transfer','Payment','Fee','Interest','Refund']
valid_channels  = ['Online','Mobile','ATM','Branch','Phone']
valid_statuses  = ['Completed','Pending','Failed']
max_amount      = 1_000_000
min_date        = '2015-01-01'
max_date        = '2026-12-31'

# Add one boolean flag per quality rule so every bad record keeps all of its failure signals
df_checked = df_dirty     .withColumn('fail_null_txn_id',   F.col('TransactionID').isNull() | (F.col('TransactionID') == ''))     .withColumn('fail_null_account',  F.col('AccountID').isNull()     | (F.col('AccountID') == ''))     .withColumn('fail_null_customer', F.col('CustomerID').isNull()    | (F.col('CustomerID') == ''))     .withColumn('fail_amount_neg',    F.col('Amount') <= 0)     .withColumn('fail_amount_max',    F.col('Amount') > max_amount)     .withColumn('fail_date_format',   F.to_date(F.col('TransactionDate'), 'yyyy-MM-dd').isNull())     .withColumn('fail_future_date',   F.to_date(F.col('TransactionDate'), 'yyyy-MM-dd') > F.lit(max_date))     .withColumn('fail_txn_type',     ~F.col('TransactionType').isin(valid_txn_types))     .withColumn('fail_channel',      ~F.col('Channel').isin(valid_channels))     .withColumn('fail_status',       ~F.col('Status').isin(valid_statuses))

# Collect the temporary failure-flag columns so they can be summarized generically
fail_cols = [c for c in df_checked.columns if c.startswith('fail_')]

# Combine all triggered rule names into one readable failure-reasons string per row
reason_expr = F.concat_ws(' | ', *[
    F.when(F.col(c), F.lit(c.replace('fail_','')))
    for c in fail_cols
])

# Add the readable failure text and initialize an overall pass/fail flag for each record
df_checked = df_checked     .withColumn('_failure_reasons', reason_expr)     .withColumn('_has_failures', F.lit(False))

# Roll up the individual rule flags into one master indicator showing whether a row failed any check
for c in fail_cols:
    df_checked = df_checked.withColumn('_has_failures', F.col('_has_failures') | F.col(c))

# Remove the intermediate boolean columns now that the summary flags are ready
df_checked = df_checked.drop(*fail_cols)

# Print the validation totals so the notebook shows how many rows passed or failed
print(f'Total checked: {df_checked.count()}')
print(f'Passed:        {df_checked.filter(~F.col("_has_failures")).count()}')
print(f'Failed:        {df_checked.filter( F.col("_has_failures")).count()}')

## Step 2 — Route: Clean → Silver, Bad → Dead-Letter

In [ ]:
# Keep only the records that passed every rule and stamp when they were validated
df_clean = df_checked.filter(~F.col('_has_failures'))     .drop('_has_failures','_failure_reasons')     .withColumn('_validated_at', F.current_timestamp())

# Write the clean records to the validated silver table for downstream consumption
df_clean.write.format('delta').mode('append').saveAsTable('silver_transactions_validated')
print(f'✅ Clean records written to silver_transactions_validated: {df_clean.count()}')

# Keep the failing records, add quarantine metadata, and preserve the failure reasons for debugging
df_bad = df_checked.filter(F.col('_has_failures'))     .withColumn('_quarantined_at', F.current_timestamp())     .withColumn('_source',         F.lit('dirty_transactions.csv'))

# Write invalid rows to the dead-letter table so they can be reviewed without blocking valid data
df_bad.write.format('delta').mode('append').saveAsTable('dead_letter_transactions')
print(f'⚠️  Bad records quarantined to dead_letter_transactions: {df_bad.count()}')

## Step 3 — Review Dead-Letter Table

In [ ]:
%%sql
-- Review the quarantined transactions together with the failure reasons captured for each record
SELECT TransactionID, AccountID, Amount, TransactionDate,
       TransactionType, Channel, Status,
       _failure_reasons
FROM dead_letter_transactions
ORDER BY _quarantined_at DESC

## Step 4 — Quality Summary Dashboard

In [ ]:
%%sql
-- Summarize which rule failures appear most often in the dead-letter table
-- Quality summary: which rules triggered most?
SELECT
    _failure_reasons AS FailureReason,
    COUNT(*)          AS Count
FROM dead_letter_transactions
GROUP BY _failure_reasons
ORDER BY Count DESC

## Step 5 — Quality Score KPI

In [ ]:
# Calculate the key quality metrics needed for the final scorecard
total   = df_checked.count()
passed  = df_clean.count()
failed  = df_bad.count()
score   = round(passed / total * 100, 1)

# Print a simple report header and the pass/fail counts for this validation run
print('=' * 45)
print('  DATA QUALITY REPORT')
print('=' * 45)
print(f'  Total records:    {total}')
print(f'  Passed:           {passed}  ✅')
print(f'  Failed/Quarantined: {failed}  ⚠️')
print(f'  Quality Score:    {score}%')
print('  Target:           >= 95%')

# Compare the score to the target threshold and print the final quality result
status = '✅ PASS' if score >= 95 else '❌ FAIL — investigate dead-letter table'
print(f'  Result:           {status}')
print('=' * 45)